# Week 04 - Control FLow & Screening Decision Logic

**Focus:** Translating experimental decision-making into structured compputational logic using conditionals, loopls, and resuable functions.

**Skills practiced:**
- Writing multi-branch decision logic ('if/ elif/ else')
- Using logical operators ('and', 'or')
- Iterating over lists and dictionaries with 'for' loops
- Writing reusable functions with 'def' and 'return'
- Combining loops and functions into mini screening pipelines
- Producing clean, interpretable decision outputs

**Biological context:** In translational screening workflows, compound advancement decisions are based on effect size, toxicity, and pre-defined go/no-go criteria. This week formalizes that biological reasoning into reproducible Python logic, simulating a structureed hit-calling pipeline. 

In [6]:
import os

os.listdir()

['.ipynb_checkpoints', 'week_04_control_flow.ipynb']

## Day 01 - If /Elif /Else

In [7]:
# Step 1 - Define screening inputs

# Percent remyelination effect observed
remyelination_percent = 45

#Toxicity flag (True means toxic)
toxicity_flag = False

In [8]:
# Step 2 - Apply decision rules

if toxicity_flag == True:
    decision = "Reject due to toxicity"

elif remyelination_percent > 50:
    decision = "Strong effect"

elif remyelination_percent >= 20 and remyelination_percent <=50:
    decision = "Moderate effect"

else:
    decision = "Weak effect"

In [9]:
# Step 3 - Display final decision 

print("Remyelination percent:", remyelination_percent)
print("Toxicity flag:", toxicity_flag)
print("Final decision:", decision)

Remyelination percent: 45
Toxicity flag: False
Final decision: Moderate effect


In [10]:
# Refactored decision logic (cleaner ranges)

if toxicity_flag:
    decision = "Reject due to toxicity"

elif remyelination_percent > 50:
    decision = "Strong effect"

elif 20<= remyelination_percent <= 50:
    decision = "Moderate effect"

else:
    decision = "Weak effect"

## Day 02 - For Loops (Batch Evaluation)

In [11]:
# Step 1 - Define screening dataset

compounds = ["A", "B", "C", "D"]

effect_values = [12, 78, 34, 91]

toxicity_flags = [False, False, True, False]

In [12]:
# Step 2 - Loop through each compound using index

for i in range(len(compounds)):
    compound_name = compounds[i]
    remyelination_percent = effect_values[i]
    toxicity_flag = toxicity_flags[i]

    # Apply decision logic 
    if toxicity_flag:
        decision = "Reject due to toxicity"

    elif remyelination_percent > 50:
        decision = "Strong effect"

    elif 20 <= remyelination_percent <=50:
        decision = "Moderate effect"

    else:
        decision = "Weak effect"

    # Print result clearly 

    print("Compound:", compound_name)
    print("  Effect:", remyelination_percent)
    print("  Toxicity:", toxicity_flag)
    print("  Decision:", decision)
    print("------------------------------")

Compound: A
  Effect: 12
  Toxicity: False
  Decision: Weak effect
------------------------------
Compound: B
  Effect: 78
  Toxicity: False
  Decision: Strong effect
------------------------------
Compound: C
  Effect: 34
  Toxicity: True
  Decision: Reject due to toxicity
------------------------------
Compound: D
  Effect: 91
  Toxicity: False
  Decision: Strong effect
------------------------------


**This works, but parallel lists are fragile because they rely on same length, same ordering and prone to insertion mistakes - so the real screening data is rarely structured this way.** 

In [13]:
# Step 1 - Create structured screening data using a dictionary 

screening_data = {
    "A": {"effect": 12, "toxicity": False},
    "B": {"effect": 78, "toxicity": False},
    "C": {"effect": 34, "toxicity": True},
    "D": {"effect": 91, "toxicity": False}
}

In [14]:
# Step 2 - Loop through structured dictionary 

for compound_name in screening_data:
    remyelination_percent = screening_data[compound_name]["effect"]
    toxicity_flag = screening_data[compound_name]["toxicity"]

    # Apply decision logic 
    if toxicity_flag:
        decision = "Reject due to toxicity"

    elif remyelination_percent > 50:
        decision = "Strong effect"

    elif 20 <= remyelination_percent <= 50:
        decision = "Moderate effect"

    else:
        decision = "Weak effect"

    print(compound_name, "->", decision)

A -> Weak effect
B -> Strong effect
C -> Reject due to toxicity
D -> Strong effect


## Day 03 - Functions (Reusable Decision Logic)

In [18]:
# Step 1 - Define reusable decision function 

def evaluate_compound(effect_value, toxicity_flag):

    if toxicity_flag:
        return "Reject due to toxicity"

    elif effect_value > 50:
        return "Strong effect"

    elif 20 <= effect_value <= 50:
        return "Moderate effect"

    else:
        return "Weak effect"

In [19]:
# Step 2 - Run the storage Logic 

decision_results = {}

for compound_name in screening_data: 

    effect_value = screening_data[compound_name]["effect"]
    toxicity_flag = screening_data[compound_name]["toxicity"]

    decision = evaluate_compound(effect_value, toxicity_flag)

    decision_results[compound_name] = decision

In [20]:
decision_results

{'A': 'Weak effect',
 'B': 'Strong effect',
 'C': 'Reject due to toxicity',
 'D': 'Strong effect'}

In [24]:
acceptable_count = 0

for decision in decision_results.values():
    if decision in ["Strong effect", "Moderate effect"]:
        acceptable_count += 1 

acceptable_count

2

In [30]:
def count_acceptable(decision_dict):
    acceptable_count = 0

    for decision in decision_dict.values():
        if decision in ["Strong effect", "Moderate effect"]:
            acceptable_count += 1

    return acceptable_count

In [31]:
count_acceptable(decision_results)

2

In [32]:
def evaluate_compound(effect_value, toxicity_flag, strong_threshold = 50, moderate_threshold=20):

    if toxicity_flag:
        return "Reject due to toxicity"

    elif effect_value > strong_threshold:
        return "Strong effect"

    elif moderate_threshold <= effect_value <= strong_threshold:
        return "Moderate effect"

    else:
        return "Weak effect"

In [35]:
evaluate_compound(65, False, strong_threshold=60, moderate_threshold=30)

'Strong effect'

**if toxicity is always a hard stop, it should remain structurally prioritized in the function because it makes the logic clear, prevents accidental "but it's strong!" overrides, and it reflects how real go/no-go criteria work.**

In [36]:
def evaluate_compound(effect_value, toxicity_flag,
                      strong_threshold=50,
                      moderate_threshold=20):
    
    # Hard stop: toxicity overrides efficacy
    if toxicity_flag:
        return "Reject due to toxicity"
    
    if effect_value > strong_threshold:
        return "Strong effect"
    
    if moderate_threshold <= effect_value <= strong_threshold:
        return "Moderate effect"
    
    return "Weak effect"


In [37]:
evaluate_compound(65, False, strong_threshold=60, moderate_threshold=30)

'Strong effect'

## Refresher - Core Function + Batch Evaluation

In [2]:
# Step 1: Define the decision function 

def evaluate_compound(effect_value, toxicity_flag,
                      strong_threshold=50,
                      moderate_threshold=20):
    # Hard stop: toxicity overrides everything 
    if toxicity_flag: 
        return "Reject due to toxicity"

    # Efficacy categories
    if effect_value> strong_threshold: 
        return "Strong effect"

    if moderate_threshold <= effect_value <= strong_threshold:
        return "Moderate effect"

    return "Weak effect"

In [3]:
# Step 2: Define screening data (explicit)

screening_data = {
    "A": {"effect": 12, "toxicity": False}, 
    "B": {"effect": 78, "toxicity": False},
    "C": {"effect": 34, "toxicity": True},
    "D": {"effect": 91, "toxicity": False}
}

In [4]:
# Step 3 - Loop through compounds + store reults

decision_results = {}

for compound_name in screening_data: 

    effect_value = screening_data[compound_name]["effect"]
    toxicity_flag = screening_data[compound_name]["toxicity"]

    decision = evaluate_compound(
        effect_value=effect_value, 
        toxicity_flag=toxicity_flag, 
        strong_threshold=50,
        moderate_threshold=20
    )
    decision_results[compound_name] = decision 

decision_results

{'A': 'Weak effect',
 'B': 'Strong effect',
 'C': 'Reject due to toxicity',
 'D': 'Strong effect'}

In [6]:
def classify_effect(effect_value):
    if effect_value >70:
        return "High"

    else: 
        return "Low"

In [7]:
classify_effect(80)

'High'

In [8]:
classify_effect(45)

'Low'

In [14]:
import pandas as pd

data = {
    "compound": ["A", "B", "C", "D", "E"],
    "effect": [55, 38, 12, 29, 45], 
    "toxicity": [10, 12, 8, 25, 18]
}

df = pd.DataFrame(data)

df["decision"] = {}

for i in range(len(df)):
    eff = df["effect"][i]
    tox = df["toxicity"][i]

    if eff >=40 and tox <= 15: 
        df["decision"][i] = "ADVANCE"

    elif eff >=25 and tox <= 20:
        df["decision"][i] = "HOLD"

    elif eff < 25 and tox >20:
        df["decision"][i] = "DROP"

df["hit"] = df["decision"] = "ADVANCE"

df


C:\Users\asul0\AppData\Local\Temp\ipykernel_489016\3530150483.py:18: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df["decision"][i] = "ADVANCE"
C:\Users\asul0\AppData\Local\Temp\ipykernel_489016\3530150483.py:18: SettingWithCopyWarning: 
A 

,compound,effect,toxicity,decision,hit
0,A,55,10,ADVANCE,ADVANCE
1,B,38,12,ADVANCE,ADVANCE
2,C,12,8,ADVANCE,ADVANCE
3,D,29,25,ADVANCE,ADVANCE
4,E,45,18,ADVANCE,ADVANCE


In [1]:
import pandas as pd

data = {
    "compound": ["A", "B", "C", "D", "E"],
    "effect": [55, 38, 12, 29, 45],
    "toxicity": [10, 12, 8, 25, 18]
}

df = pd.DataFrame(data)

In [2]:
def classify_compound(eff, tox):
    if tox >25:
        return "DROP"
    elif eff >=50:
        return "ADVANCE"
    elif eff >=30:
        return "HOLD"
    else:
        return "DROP"

In [3]:
df["decision"] = df.apply(
    lambda row: classify_compound(row["effect"], row["toxicity"]),
    axis=1
)

In [5]:
df["hit"] = df["decision"] == "ADVANCE"

In [6]:
df

,compound,effect,toxicity,decision,hit
0,A,55,10,ADVANCE,True
1,B,38,12,HOLD,False
2,C,12,8,DROP,False
3,D,29,25,DROP,False
4,E,45,18,HOLD,False


**Vectorized approach**

In [7]:
import pandas as pd 

data = {
    "compound": ["A", "B", "C", "D", "E"],
    "effect": [55, 38, 12, 29, 45],
    "toxicity": [10, 12, 8, 25, 18]
}
df=pd.DataFrame(data)

#-----------------------------------
# Step 1) Start with a default decision for everyone 
#-----------------------------------
df["decision"] = "DROP"

#-----------------------------------
# Step 2) Build boolean masks (True/False arrays)
# These masks identify which rows meet each condition 
#------------------------------------
safe_mask = df["toxicity"] <=25

advance_mask = safe_mask & (df["effect"] >= 50)

hold_mask = safe_mask & (df["effect"] >= 30) & (df["effect"] < 50)

#-------------------------------------
# Step 3) Assign decisions using the masks 
# Order matters: ADVANCE first, then HOLD 
#-------------------------------------
df.loc[advance_mask, "decision"] = "ADVANCE"
df.loc[hold_mask, "decision"] = "HOLD"


#--------------------------------------
# Step 4) Make hit column (boolean label)
#--------------------------------------
df["hit"] = df["decision"] == "ADVANCE"

df

,compound,effect,toxicity,decision,hit
0,A,55,10,ADVANCE,True
1,B,38,12,HOLD,False
2,C,12,8,DROP,False
3,D,29,25,DROP,False
4,E,45,18,HOLD,False
